# Featurizer Tutorial: Custom Primitives

This notebook demonstrates how to extend Featurizer with custom aggregation and transformation primitives using a financial analytics scenario.

> **Note:** Since the primitives expansion (commit `6b25339`), several primitives demonstrated here — Median, P95, Range, Log, ZScore — are now built-in. This tutorial is retained to show the **registration pattern** for creating truly custom primitives.

## 1. Built-in Primitives That Cover Common Needs

Before writing custom primitives, check if your need is already covered. Featurizer now ships with 43 aggregations and 71 transformations.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

from featurizer.primitives.utils import list_aggregations, list_transformations

aggs = sorted(list_aggregations())
transforms = sorted(list_transformations())

print(f"Built-in aggregations ({len(aggs)}):")
print(f"  {', '.join(aggs)}")
print(f"\nBuilt-in transformations ({len(transforms)}):")
print(f"  {', '.join(transforms)}")

Built-in aggregations (69):
  acf_1, age_in_system, all, any, bbox_area, burstiness, cosinor_amplitude_weekly, count, cross_type_latency, cv, distance_travelled, entropy, event_rate, first_passage_time, gap_cv, gap_max, gap_mean, gap_min, gap_stddev, geometric_mean, gini, harmonic_mean, hhi, inter_event_hazard_proxy, iqr, kl_drift, kurtosis, longest_streak, markov_conditional_entropy, max, max_transition_prob, mean, mean_deviation, median, median_absolute_deviation, min, min_max_scale, mode, ngram_2_freq, ngram_3_freq, nunique, p10, p25, p75, p90, p95, p99, radius_of_gyration, range, recency, recurrence_interval, rework_count, right_censoring_indicator, sequence_entropy, skewness, spatial_std, state_volatility, stddev, sum, tenure, theil, time_in_current_state, time_span, transition_matrix_summary, trimmed_mean_10, variance, variance_ratio, wasserstein_drift, z_score

Built-in transformations (83):
  abs, avg_word_length, caps_ratio, cbrt, cdf, ceil, century, cross_entity_percentile, c

### Built-in Equivalents of This Example's Custom Primitives

| Custom (this example) | Built-in name | Type |
|---|---|---|
| `Median` | `median` | aggregation |
| `Percentile95` | `p95` | aggregation |
| `Range` | `range` | aggregation |
| `Log` | `ln` | transformation |
| `ZScore` | `cross_entity_zscore` | transformation |

To use built-in versions, no custom code is needed — just reference them by name.

## 2. The Custom Registration Pattern

When you need a truly custom primitive (e.g., domain-specific financial metrics), Featurizer provides a registration API. Let's walk through the pattern.

In [2]:
with open("custom_primitives.py") as f:
    print(f.read())

"""Custom aggregation and transformation primitives for financial analytics.

These demonstrate the current primitive API and emit PostgreSQL-valid SQL:

- An **aggregation** subclasses ``Aggregator`` (or a richer base such as
  ``OrderedSetAggregator``) and overrides ``_build_aggregate_expression`` to
  return the SQL aggregate; the base wraps it into a ``Feature``. Interval
  variants pass ``interval`` so the expression can add the ``daterange`` filter.
- A **transformation** subclasses ``Transformer`` and overrides
  ``_build_transformer_call`` to return a scalar SQL expression over the
  current row; the base wraps it into a ``Feature``.

Register *instances* (not classes) via ``register_aggregation`` /
``register_transformer``, then select them by name in ``config.yaml``.
"""

from featurizer.primitives.aggregations import Aggregator, OrderedSetAggregator
from featurizer.primitives.transformations import Transformer
from featurizer.primitives.utils import register_aggregation, reg

### Key Steps:
1. **Subclass** `Aggregation` or `Transformation` from `featurizer.primitives.abstractions`
2. **Implement** `to_sql(feature, alias)` returning a SQL expression
3. **Register** via `register_aggregation()` or `register_transformation()`
4. **Call** `register_all_custom_primitives()` before creating the Featurizer

**Important**: Transformations must return new `Feature` instances (never mutate) to preserve hashing semantics.

## 3. Setup and Registration

In [3]:
import sys
from pathlib import Path

# This tutorial is database-free: it loads the config and inspects the
# synthesized features and the generated SQL — none of which touch a
# database. To execute the features against PostgreSQL, run
# `just example 04-custom-primitives` (create_data.py + run_example.py).
sys.path.insert(0, str(Path.cwd().parent.parent))

# Custom primitives must be registered BEFORE creating the Featurizer.
from custom_primitives import register_all_custom_primitives

register_all_custom_primitives()

✓ Registered custom primitives:
  Aggregations: range, p95
  Transformations: log1p, zscore, bin


## 4. Create Featurizer with Custom Primitives

In [4]:
from featurizer import Featurizer

featurizer = Featurizer("config.yaml")

print(f"Target entity: {featurizer.target.alias}")
print(f"Max depth: {featurizer.max_depth}")
print(f"Intervals: {featurizer.intervals}")

target_features = featurizer.features[featurizer.target.alias]
print(f"Generated features: {len(target_features)}")

2026-06-21 13:29:57.017 | DEBUG    | featurizer.planner:plan:230 - Starting feature build for target accounts


2026-06-21 13:29:57.018 | DEBUG    | featurizer.planner:_build_features:256 - build_features(accounts) depth=0


2026-06-21 13:29:57.018 | DEBUG    | featurizer.planner:_build_features:256 - build_features(transactions) depth=1


2026-06-21 13:29:57.018 | INFO     | featurizer.planner:_build_features:276 - Maximum recursion depth reached at depth 2; materializing transactions without traversing further.


2026-06-21 13:29:57.018 | DEBUG    | featurizer.planner:_build_aggregations:1014 - Processing backward relationship Entity(accounts).account_id -> Entity(transactions).account_id


2026-06-21 13:29:57.019 | WARNING  | featurizer.planner:_apply_direct_roles:1155 - Direct variable 'accounts.account_type' (type: categorical) will pass through as a raw string column and is likely to crash a downstream encoder. Set role: categorical (with a declared vocabulary or a PostgreSQL ENUM) to one-hot encode it, or role: identifier to exclude it.


Target entity: accounts
Max depth: 2
Intervals: ['P7D', 'P30D']
Generated features: 187


## 5. Examining Features from Custom Primitives

Let's see which features use our custom primitives.

In [5]:
custom_names = ["median", "p95", "range", "log", "zscore", "bin"]
custom_features = [
    f for f in target_features if any(prim in f.name.lower() for prim in custom_names)
]

print(f"Features using custom primitives: {len(custom_features)}")
for f in sorted(custom_features, key=lambda x: x.name)[:15]:
    print(f"  - {f.name}")

Features using custom primitives: 171
  - "BIN(accounts.COUNT(transactions.transaction_date))"
  - "BIN(accounts.COUNT(transactions.transaction_date|interval=P30D))"
  - "BIN(accounts.COUNT(transactions.transaction_date|interval=P7D))"
  - "BIN(accounts.COUNT(transactions.transaction_id))"
  - "BIN(accounts.COUNT(transactions.transaction_id|interval=P30D))"
  - "BIN(accounts.COUNT(transactions.transaction_id|interval=P7D))"
  - "BIN(accounts.COUNT(transactions.transaction_type))"
  - "BIN(accounts.COUNT(transactions.transaction_type|interval=P30D))"
  - "BIN(accounts.COUNT(transactions.transaction_type|interval=P7D))"
  - "BIN(accounts.MEAN(transactions.BIN(transactions.amount)))"
  - "BIN(accounts.MEAN(transactions.BIN(transactions.amount)|interval=P30D))"
  - "BIN(accounts.MEAN(transactions.BIN(transactions.amount)|interval=P7D))"
  - "BIN(accounts.MEAN(transactions.LOG1P(transactions.amount)))"
  - "BIN(accounts.MEAN(transactions.LOG1P(transactions.amount)|interval=P30D))"
  - "BIN(

## 6. SQL Comparison: Custom vs Built-in

Let's compare the SQL generated by a custom primitive vs its built-in equivalent.

In [6]:
sql = featurizer.query
print("Generated SQL (with custom primitives):")
print("=" * 80)
print(sql)
print("=" * 80)

2026-06-21 13:29:57.026 | DEBUG    | featurizer.sql:render:40 - Rendered SQL for target 'accounts': 5 CTEs, 60572 chars


Generated SQL (with custom primitives):

        select aod.as_of_date, t.*
        from as_of_dates as aod
        cross join lateral (

        with

        
        -- sythetize aggregations and direct features for transactions
        transactions_synth as (
        select
        transactions.transaction_id, transactions.transaction_date, transactions.account_id, amount, transaction_type
        from transactions
        
        
        )
        ,
        -- transform transactions
        transactions_transform as (
        select
        transaction_id, transaction_date, account_id, case when amount is null then null else cast(floor(5 * (amount - min(amount) over ()) / nullif(max(amount) over () - min(amount) over (), 0)) as integer) end as "BIN(transactions.amount)" , case when amount + 1 > 0 then ln(amount + 1) end as "LOG1P(transactions.amount)" , ((amount - avg(amount) over ()) / nullif(stddev(amount) over (), 0)) as "ZSCORE(transactions.amount)" , amount as amount, trans

## 7. Summary

In this tutorial, we learned:

1. **Check built-ins first**: Featurizer now ships with 43 aggregations and 71 transformations covering most common needs
2. **Registration pattern**: How to create custom primitives when built-ins aren't enough
3. **Implementation**: Subclass, implement `to_sql()`, register, use
4. **Best practices**: Always return new `Feature` instances from transformers

### When to Write Custom Primitives
- Domain-specific metrics (e.g., financial ratios, medical scores)
- Complex SQL expressions not covered by built-ins
- Organization-specific feature definitions

In [7]:
print("Custom Primitives Summary")
print("=" * 40)
print(f"Built-in aggregations: {len(aggs)}")
print(f"Built-in transformations: {len(transforms)}")
print(f"Custom aggregations: 3 (median, p95, range)")
print(f"Custom transformations: 3 (log, zscore, bin)")
print(f"Total features generated: {len(target_features)}")

Custom Primitives Summary
Built-in aggregations: 69
Built-in transformations: 83
Custom aggregations: 3 (median, p95, range)
Custom transformations: 3 (log, zscore, bin)
Total features generated: 187
